# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a walk-through for loading and exploring the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is defined using the [Croissant schema](https://mlcommons.org/croissant/) and is referenced by a schema URL.

In [ ]:
# Install mlcroissant if not already present
!pip install -q mlcroissant

## 1. Data Loading
Load dataset metadata and records with the `mlcroissant` API.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# The Croissant schema URL for this dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Record sets describe main tables of the dataset, with their fields. We'll enumerate record sets, their `@id`'s, and key fields for further extraction.

Each item is referenced by its `@id` for reliable, schema-based referencing.

In [ ]:
# Get all record sets in the dataset
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
else:
    # fallback for attribute name differences
    record_sets = getattr(metadata, 'recordSet', [])

if not record_sets:
    # Try secondary method in case record_sets are not attached to metadata
    # Attempt discovery from the dataset itself
    record_sets = dataset.record_sets # May be empty if dataset supplies none

# List record sets and their ids/fields
discovered_record_sets = []
print('Available record sets:')
for rs in dataset.record_sets:
    print(f"- @id: {rs['@id']}  name: {rs.get('name', '')}")
    print("  Fields:")
    for f in rs.get('fields', []):
        print(f"    - @id: {f['@id']}  name: {f.get('name','')}")
    discovered_record_sets.append(rs['@id'])

if not discovered_record_sets:
    print("No record sets discovered. Check dataset schema.")

## 3. Data Extraction
We'll select one or more record sets using their `@id` fields for data extraction. The field `@id`s will be used to select or reference columns in the DataFrame.

Below, we load data from each record set into a pandas DataFrame.

In [ ]:
# Use the discovered record set @id's from above for extraction
dataframes = {}
for record_set_id in discovered_record_sets:
    print(f"\nLoading records for record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records. Columns (field @id's):\n", list(df.columns))
        # Display first few records
        display(df.head())
    else:
        print(f"No records found for record set {record_set_id}.")
if not dataframes:
    print("No tabular data extracted. Check the dataset definition and schema.")

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field (via its `@id` as listed in the DataFrame columns), filter by a threshold, normalize the field, and group (if possible) by a categorical field (using its `@id`).

In [ ]:
# Pick a record set and fields by their @id (these must be changed to match actual field ids from output above)
# For illustration: Replace these with real values in your use

# ===> USER needs to look up actual field @ids in Data Overview (Step 2) output <===
example_record_set_id = None
example_numeric_field_id = None
example_group_field_id = None  # set this if you want to group

# Try pick the first DataFrame, guess numeric field:
if dataframes:
    example_record_set_id = next(iter(dataframes))
    df = dataframes[example_record_set_id]
    # Try to guess a numeric field by checking dtype or column name
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            example_numeric_field_id = col
            break
    if not example_numeric_field_id:
        # Try by name heuristic
        for col in df.columns:
            if any(x in col.lower() for x in ["age", "year", "count", "interval", "days"]):
                example_numeric_field_id = col
                break
    # Try guess group field by name
    for col in df.columns:
        if any(x in col.lower() for x in ["sex", "gender", "group", "category", "site", "location"]):
            example_group_field_id = col
            break

    print(f"Sample chosen for EDA: record set @id = {example_record_set_id}")
    if example_numeric_field_id:
        print(f"Numeric field @id for filtering: {example_numeric_field_id}")
    if example_group_field_id:
        print(f"Grouping field @id: {example_group_field_id}")
else:
    print("No data available for EDA. Please check record set extraction above.")

# Proceed with filtering, normalization, grouping if numeric_field_id is found
if dataframes and example_record_set_id and example_numeric_field_id:
    df = dataframes[example_record_set_id]
    # Drop NA for field to avoid errors
    df = df.dropna(subset=[example_numeric_field_id])
    # Try threshold as mean
    threshold = df[example_numeric_field_id].mean()
    filtered_df = df[df[example_numeric_field_id] > threshold]
    print(f"Filtered records where {example_numeric_field_id} > {threshold:.2f}")
    display(filtered_df.head())
    
    # Normalize the numeric field
    normalized_col = f"{example_numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[example_numeric_field_id] - filtered_df[example_numeric_field_id].mean()) / filtered_df[example_numeric_field_id].std()
    print(f"First 5 normalized {example_numeric_field_id} values:")
    display(filtered_df[[example_numeric_field_id, normalized_col]].head())
    
    # Grouping by group field if available:
    if example_group_field_id and example_group_field_id in filtered_df.columns:
        print(f"\nMean of normalized {example_numeric_field_id} by {example_group_field_id}:")
        group_means = filtered_df.groupby(example_group_field_id)[normalized_col].mean().reset_index()
        display(group_means)
else:
    print("(No EDA performed: suitable numeric field could not be identified automatically. Please fill in '@id's manually based on schema above.")

## 5. Visualization
Visualize the distribution of the chosen normalized numeric field, and if applicable, stratify by a chosen group field.

In [ ]:
import matplotlib.pyplot as plt

if dataframes and example_record_set_id and example_numeric_field_id:
    filtered_df = dataframes[example_record_set_id]
    fig, ax = plt.subplots(figsize=(8,4))
    normalized_col = f"{example_numeric_field_id}_normalized"
    if normalized_col in filtered_df.columns:
        series_to_plot = filtered_df[normalized_col]
    else:
        series_to_plot = filtered_df[example_numeric_field_id]
    ax.hist(series_to_plot.dropna(), bins=10, alpha=0.7)
    ax.set_title(f"Distribution of {normalized_col if normalized_col in filtered_df else example_numeric_field_id}")
    ax.set_xlabel(normalized_col if normalized_col in filtered_df else example_numeric_field_id)
    ax.set_ylabel('Count')
    plt.show()
    # Optionally boxplot/grouped by category if available
    if example_group_field_id and example_group_field_id in filtered_df.columns:
        plt.figure(figsize=(8,5))
        filtered_df.boxplot(column=example_numeric_field_id, by=example_group_field_id)
        plt.title(f"{example_numeric_field_id} by {example_group_field_id}")
        plt.suptitle('')
        plt.xlabel(example_group_field_id)
        plt.ylabel(example_numeric_field_id)
        plt.show()

## 6. Conclusion
- The dataset was successfully loaded using the Croissant schema via `mlcroissant`.
- We identified record sets and referenced all entities using their `@id`s for reproducibility and provenance.
- Data was extracted into pandas DataFrames and underwent simple filtering, normalization, grouping, and visualization.
- For additional analysis, inspect field `@id` assignments from the data overview and update corresponding field references.

<i>Notebook generated for FAIR<sup>2</sup> Croissant dataset processing. Please adjust field `@id`s as necessary for specific analysis.</i>